# Cache Report Viewer

Scans through the per-repo JSON cache files produced by the pipeline and
displays all deprecated libraries, semgrep findings, and scan errors with dates.

In [ ]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
from IPython.display import display, Markdown

CACHE_DIR = Path('./results/cache')

# Load all cache files
all_findings = []
all_vulns = []
all_errors = []
cache_files = sorted(CACHE_DIR.glob('*.json')) if CACHE_DIR.exists() else []

for f in cache_files:
    try:
        data = json.loads(f.read_text(encoding='utf-8'))
    except Exception:
        continue
    all_findings.extend(data.get('findings', []))
    all_vulns.extend(data.get('vulns', []))
    for e in data.get('errors', []):
        if isinstance(e, list) and len(e) >= 3:
            all_errors.append({'repo_name': e[0], 'stage': e[1], 'error': e[2]})

print(f'Loaded {len(cache_files)} cache files')
print(f'  {len(all_findings):,} semgrep findings')
print(f'  {len(all_vulns):,} deprecated dependencies')
print(f'  {len(all_errors):,} scan errors')

if not cache_files:
    print('\nNo cache files found in', CACHE_DIR)
    print('Run the pipeline first to generate data.')

## Deprecated Dependencies

In [ ]:
if all_vulns:
    vulns_df = pd.DataFrame(all_vulns)
    display_cols = ['repo_name', 'interval_start', 'interval_end', 'ecosystem',
                    'package', 'version', 'vulnerability_id', 'severity',
                    'description', 'fix_versions']
    vulns_df = vulns_df[[c for c in display_cols if c in vulns_df.columns]]
    vulns_df = vulns_df.sort_values('interval_start').reset_index(drop=True)

    print(f'Total: {len(vulns_df)} vulnerable dependencies\n')

    # Summary by ecosystem
    if 'ecosystem' in vulns_df.columns:
        print('By ecosystem:')
        display(vulns_df['ecosystem'].value_counts().to_frame('count'))

    # Summary by severity
    if 'severity' in vulns_df.columns:
        print('\nBy severity:')
        display(vulns_df['severity'].value_counts().to_frame('count'))

    # Top vulnerable packages
    if 'package' in vulns_df.columns:
        print('\nTop 20 vulnerable packages:')
        display(vulns_df['package'].value_counts().head(20).to_frame('count'))

    print('\nFull table:')
    display(vulns_df)
else:
    print('No deprecated dependencies found.')

## Semgrep Findings

In [ ]:
if all_findings:
    findings_df = pd.DataFrame(all_findings)
    display_cols = ['repo_name', 'interval_start', 'interval_end', 'severity',
                    'rule_id', 'cwe', 'owasp', 'confidence', 'category',
                    'message', 'file', 'line_start', 'line_end']
    findings_df = findings_df[[c for c in display_cols if c in findings_df.columns]]
    findings_df = findings_df.sort_values('interval_start').reset_index(drop=True)

    print(f'Total: {len(findings_df)} semgrep findings\n')

    # Summary by severity
    if 'severity' in findings_df.columns:
        print('By severity:')
        display(findings_df['severity'].value_counts().to_frame('count'))

    # Summary by CWE
    if 'cwe' in findings_df.columns:
        print('\nTop 20 CWEs:')
        display(findings_df['cwe'].value_counts().head(20).to_frame('count'))

    # Summary by rule
    if 'rule_id' in findings_df.columns:
        print('\nTop 20 rules:')
        display(findings_df['rule_id'].value_counts().head(20).to_frame('count'))

    print('\nFull table:')
    display(findings_df)
else:
    print('No semgrep findings found.')

## Scan Errors

In [ ]:
if all_errors:
    errors_df = pd.DataFrame(all_errors)
    errors_df = errors_df.reset_index(drop=True)

    print(f'Total: {len(errors_df)} errors\n')

    print('By stage:')
    display(errors_df['stage'].value_counts().to_frame('count'))

    print('\nFull table:')
    display(errors_df)
else:
    print('No scan errors found.')

## Summary

In [ ]:
if cache_files:
    # Date range
    dates = set()
    for f in all_findings:
        if f.get('interval_start'): dates.add(f['interval_start'])
    for v in all_vulns:
        if v.get('interval_start'): dates.add(v['interval_start'])

    date_range = f'{min(dates)} to {max(dates)}' if dates else 'N/A'

    # Repos with findings vs clean
    repos_with_findings = len({f['repo_name'] for f in all_findings if f.get('repo_name')})
    repos_with_vulns = len({v['repo_name'] for v in all_vulns if v.get('repo_name')})

    summary = {
        'Cache files': f'{len(cache_files):,}',
        'Date range': date_range,
        'Semgrep findings': f'{len(all_findings):,}',
        'Repos with findings': f'{repos_with_findings:,}',
        'Deprecated deps': f'{len(all_vulns):,}',
        'Repos with dep vulns': f'{repos_with_vulns:,}',
        'Scan errors': f'{len(all_errors):,}',
    }
    display(pd.DataFrame(summary.items(), columns=['Metric', 'Value']))
else:
    print('No data to summarize.')

## Rebuild CSVs from Cache

Run this cell to rebuild `semgrep_findings.csv`, `deprecated_dependencies.csv`, and `scan_errors.csv`
entirely from the cache JSON files. Use this to fix any CSV-cache discrepancy (e.g. after a crash
or if fields with newlines caused CSV parse errors).

In [ ]:
import csv, os, shutil
from datetime import datetime

RESULTS_DIR = Path('./results')
FINDINGS_CSV = RESULTS_DIR / 'semgrep_findings.csv'
DEPS_CSV     = RESULTS_DIR / 'deprecated_dependencies.csv'
ERRORS_CSV   = RESULTS_DIR / 'scan_errors.csv'

FINDING_FIELDS = [
    'repo_name', 'language', 'stars', 'interval_start', 'interval_end',
    'rule_id', 'severity', 'message', 'file', 'line_start', 'line_end',
    'cwe', 'owasp', 'cve', 'confidence', 'category',
]
DEP_FIELDS = [
    'repo_name', 'language', 'stars', 'interval_start', 'interval_end',
    'package', 'version', 'vulnerability_id', 'severity',
    'description', 'ecosystem', 'fix_versions',
]

def _sanitize(val):
    """Strip newlines/carriage returns that break CSV parsing."""
    if isinstance(val, str):
        return val.replace('\n', ' ').replace('\r', '')
    return val

# Back up existing CSVs
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
for p in [FINDINGS_CSV, DEPS_CSV, ERRORS_CSV]:
    if p.exists() and p.stat().st_size > 0:
        backup = p.with_suffix(f'.pre_rebuild.{ts}.csv')
        shutil.copy2(str(p), str(backup))
        print(f'Backed up {p.name} -> {backup.name}')

# Rebuild from cache
findings_count = 0
vulns_count = 0
errors_count = 0

with open(FINDINGS_CSV, 'w', newline='', encoding='utf-8') as ff, \
     open(DEPS_CSV, 'w', newline='', encoding='utf-8') as df, \
     open(ERRORS_CSV, 'w', newline='', encoding='utf-8') as ef:

    fw = csv.DictWriter(ff, fieldnames=FINDING_FIELDS)
    dw = csv.DictWriter(df, fieldnames=DEP_FIELDS)
    ew = csv.writer(ef)

    fw.writeheader()
    dw.writeheader()
    ew.writerow(['repo_name', 'stage', 'error'])

    for cache_file in cache_files:
        try:
            data = json.loads(cache_file.read_text(encoding='utf-8'))
        except Exception:
            continue

        for f in data.get('findings', []):
            row = {k: _sanitize(f.get(k, '')) for k in FINDING_FIELDS}
            fw.writerow(row)
            findings_count += 1

        for v in data.get('vulns', []):
            row = {k: _sanitize(v.get(k, '')) for k in DEP_FIELDS}
            dw.writerow(row)
            vulns_count += 1

        for e in data.get('errors', []):
            if isinstance(e, list) and len(e) >= 3:
                ew.writerow([_sanitize(e[0]), _sanitize(e[1]), _sanitize(str(e[2])[:300])])
                errors_count += 1

print(f'\nRebuilt CSVs from {len(cache_files)} cache files:')
print(f'  {findings_count:,} semgrep findings  -> {FINDINGS_CSV.name}')
print(f'  {vulns_count:,} dep vulnerabilities -> {DEPS_CSV.name}')
print(f'  {errors_count:,} scan errors         -> {ERRORS_CSV.name}')